# NRR on TM@SxNy-graphene single-atom catalysts with Meta's UMA

This notebook computes "DFT-like" energies for the **196 TM@SxNy catalysts** of Xu et al. (Nano Research 2025)
with the universal machine-learning potential **UMA** (Meta FAIR Chemistry, `fairchem-core`).

**What it produces** (all saved to your Google Drive, so a disconnect loses nothing):

| quantity | meaning |
|---|---|
| ΔE, ΔG of 13 intermediates | *H, *N2 (end/side), *NNH (end/side), *NNH2, *N, *NH, *NH2, *NH3, *NHNH, *NHNH2, *NH2NH2 |
| U_L (distal, alternating) | limiting potential, −max ΔG of the proton–electron steps |
| ΔG*H | HER descriptor |
| Ef, U_diss | stability (formation energy vs bulk metal, dissolution potential) |

**Steps:** 1 GPU → 2 install → 3 Hugging Face login → 4 files → 5 load UMA → 6 references →
7 validation against Xu DFT → 8 choose task → 9 screening of all 196 SACs → 10 results table.

Before you start: request access to the model at **https://huggingface.co/facebook/UMA** (log in, fill the short form,
wait for approval), then create a *Read* token at https://huggingface.co/settings/tokens.

## 1. Turn on the GPU
Menu **Runtime → Change runtime type → T4 GPU → Save**. Then run the cell below (Shift+Enter).
It should print a GPU name. A CPU-only runtime works too but is ~20× slower.

In [ ]:
import sys, os, subprocess, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('No GPU found - running on CPU')

## 2. Install fairchem (UMA) and the D3 dispersion correction
Takes 2–5 minutes. If Colab asks you to **restart the session**, click *Restart* and continue from step 3.

In [ ]:
MODEL = 'uma'          # 'uma' (the goal)  |  'mace' = MACE-MPA-0 stand-in that needs no login (for testing)
if IN_COLAB:
    pkgs = ['fairchem-core', 'torch-dftd'] if MODEL == 'uma' else ['mace-torch', 'torch-dftd']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('installed')

## 3. Log in to Hugging Face (only for UMA)
Recommended: click the **key icon (Secrets)** in Colab's left bar → *Add new secret* → name `HF_TOKEN`, value = your token,
and switch on *Notebook access*. The cell reads it from there; otherwise it asks you to paste the token.

In [ ]:
if MODEL == 'uma':
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            pass
    from huggingface_hub import login
    login(token=token) if token else login()

## 4. Get the structure files
Upload **`SAC_UMA_package.zip`** to a folder **`NRR_UMA`** in *My Drive* (drag-and-drop in drive.google.com).
The cell mounts Drive (click *Allow*), unzips the package and creates the results folder `NRR_UMA/results_<task>/`.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/NRR_UMA'
    PKG = '/content/SAC_UMA_package'
    if not os.path.isdir(PKG):
        subprocess.run(['unzip', '-q', '-o', f'{WORK}/SAC_UMA_package.zip', '-d', '/content/'], check=True)
else:                                   # running on your own computer
    WORK = os.path.abspath('NRR_UMA_run')
    PKG = os.path.abspath(os.environ.get('SAC_PKG', 'SAC_UMA_package'))
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, PKG)

import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
import nrr_mlip as nm
import validate as vl

man = pd.read_csv(f'{PKG}/manifest.csv')
ref = pd.read_csv(f'{PKG}/xu_reference.csv')
print(man.groupby('tier').size())
# look at one structure (top and side view)
a = read(f'{PKG}/structures/Mo@S3N1/N2_end.vasp')
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
plot_atoms(a, ax[0], radii=0.4); plot_atoms(a, ax[1], radii=0.4, rotation='-90x'); ax[0].set_title('Mo@S3N1 + N2 (top)'); ax[1].set_title('side')
plt.show()

## 5. Load the model
UMA is one network with several **tasks** (heads trained on different DFT data):

* `oc20` – catalysis: slabs + adsorbates, **RPBE** (no D3). Gas references come from the OC20 convention
  (E(H) = ½E(H2), E(N) = ½E(N2)); it is *not* meant for isolated molecules.
* `omat` – inorganic materials, **PBE/PBE+U** (same level as Materials Project). Used for bulk metals and also tested for the SACs.

Xu et al. used **PBE-D3**, so we test both tasks against their numbers in step 7 and keep the better one.
The first call downloads the checkpoint (~1 GB).

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TASKS = ['oc20', 'omat']
calcs, d3 = {}, {}
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
if MODEL == 'uma':
    from fairchem.core import pretrained_mlip, FAIRChemCalculator
    predictor = pretrained_mlip.get_predict_unit('uma-s-1p1', device=DEVICE)
    for t in TASKS:
        calcs[t] = FAIRChemCalculator(predictor, task_name=t)
    d3 = {'oc20': TorchDFTD3Calculator(device=DEVICE, damping='zero', xc='rpbe'),
          'omat': TorchDFTD3Calculator(device=DEVICE, damping='zero', xc='pbe')}
else:   # MACE stand-in: one PBE model, so only one 'task'
    from mace.calculators import mace_mp
    TASKS = ['mace']
    calcs['mace'] = mace_mp(model='medium-mpa-0', default_dtype='float32', device=DEVICE)
    d3 = {'mace': TorchDFTD3Calculator(device=DEVICE, damping='zero', xc='pbe')}
# speed test on one structure
a = read(f'{PKG}/structures/Mo@N4/N2_end.vasp')
t0 = time.time(); E, n, ok = nm.relax(a, calcs[TASKS[0]]); dt = time.time() - t0
print(f'{TASKS[0]}: Mo@N4+N2 relaxed in {n} steps, {dt:.1f} s, E = {E:.3f} eV')

## 6. Reference calculations (≈ 5–10 min)
* gas molecules N2, H2, NH3 (used by `omat`; `oc20` uses its fixed references)
* the 7 metal-free supports (needed for the formation energy Ef)
* the 28 bulk metals, with the cell relaxed (chemical potential of the metal)

Every finished structure is appended to `results_<task>/results.jsonl` on Drive. **If Colab disconnects, just
re-run steps 1–5 and then this cell again: finished structures are skipped.**

In [ ]:
TEST_MODE = False      # True = only 2 catalysts per step (quick check of the whole notebook)
def out(task):
    d = f'{WORK}/results_{task}'; os.makedirs(d, exist_ok=True); return d

for task in TASKS:
    t0 = time.time()
    for _, r in man[man.tier == '0_reference'].iterrows():
        if TEST_MODE and r.kind == 'bulk' and r.system not in ('Mo', 'W', 'Fe'):
            continue
        nm.run_simple(r.kind, r.system, f'{PKG}/{r.file}', calcs[task], out(task), d3=d3[task],
                      cell=(r.kind == 'bulk'), fmax=0.02 if r.kind == 'bulk' else 0.05)
    print(f'{task}: references done in {time.time()-t0:.0f} s')

## 7. Validation against Xu et al. DFT (≈ 30–60 min per task on a T4)
23 catalysts (the ones highlighted by Xu + 2 random per support) with **all** intermediates, for each task.
For each structure the notebook: relaxes it (LBFGS, fmax = 0.05 eV/Å) → checks that the adsorbate stayed
on the metal → computes the vibrational ZPE and TS of the adsorbate (finite differences) → adds the D3 energy as a
separate column. For *H it also strips the H and re-relaxes the bare SAC, in case a lower-energy bare geometry exists.

In [ ]:
ADS_ALL = ['H', 'N2_end', 'N2_side', 'NNH_end', 'NNH_side', 'NNH2', 'N', 'NH', 'NH2', 'NH3', 'NHNH', 'NHNH2', 'NH2NH2']
VALID = sorted(man[(man.tier == '1_validation') & (man.kind == 'bare')].system)
if TEST_MODE:
    VALID = VALID[:2]
for task in TASKS:
    t0 = time.time()
    for i, sac in enumerate(VALID):
        nm.run_sac(sac, f'{PKG}/structures', ADS_ALL, calcs[task], out(task), d3=d3[task])
        print(f'{task} {i+1}/{len(VALID)} {sac}  ({(time.time()-t0)/60:.1f} min)', flush=True)

## 8. Compare with Xu DFT and choose the task
`MAE` = mean absolute error vs Xu (PBE-D3). `mean_shift` = systematic offset; `MAE_after_linear_fit` = error left
after a straight-line correction (slope/intercept). A model with a large shift but small `MAE_after_linear_fit`
still ranks catalysts correctly and can be calibrated. Structures where the adsorbate moved away are excluded.

In [ ]:
tables, sacs = {}, {}
for task in TASKS:
    for use_d3 in (False, True):
        key = f'{task}{"+D3" if use_d3 else ""}'
        refs = 'oc20' if task == 'oc20' else 'computed'
        ads_df, sac_df = nm.analyse(f'{out(task)}/results.jsonl', refs=refs, use_d3=use_d3)
        sac_df = vl.add_udiss(sac_df, ref)
        tables[key] = vl.compare(sac_df, ref); sacs[key] = sac_df
        print(f'\n=== {key} ==='); print(tables[key].round(3).to_string(index=False))
        vl.parity(sac_df, ref, title=key, path=f'{WORK}/parity_{key}.png'); plt.show()
# structures that did not behave
for task in TASKS:
    ads_df, _ = nm.analyse(f'{out(task)}/results.jsonl', refs='oc20' if task == 'oc20' else 'computed')
    bad = ads_df[ads_df.status != 'ok'][['system', 'kind', 'status']]
    print(f'{task}: {len(bad)} structures where the adsorbate moved/broke'); print(bad.to_string(index=False))

In [ ]:
# Pick the setting with the lowest average MAE over the adsorption quantities
score = {k: t[t.quantity.str.startswith('ΔE')].MAE.mean() for k, t in tables.items() if len(t)}
print(pd.Series(score).sort_values().round(3))
BEST = min(score, key=score.get)
TASK, USE_D3 = BEST.replace('+D3', ''), BEST.endswith('+D3')
print('-> using', TASK, 'with D3' if USE_D3 else 'without D3')

## 9. Screening all 196 catalysts (≈ 3–6 h on a T4; can be split over several sessions)
Tier 2 first (bare, *H, *N2 end/side, *NNH end/side, *NH2, *NH3 → Xu's descriptors, ΔG*H and Ef),
then tier 3 (the rest of the distal and alternating pathways). Re-running the cell continues where it stopped.
Free Colab gives a GPU for a few hours per day; just come back and re-run steps 1–5 and this cell.

In [ ]:
ALL = sorted(man[man.kind == 'bare'].system)
TIER2 = ['H', 'N2_end', 'N2_side', 'NNH_end', 'NNH_side', 'NH2', 'NH3']
if TEST_MODE:
    ALL = VALID[:2]
for ads_set, label in ((TIER2, 'tier 2'), (ADS_ALL, 'tier 3')):
    t0 = time.time()
    for i, sac in enumerate(ALL):
        nm.run_sac(sac, f'{PKG}/structures', ads_set, calcs[TASK], out(TASK), d3=d3[TASK])
        if (i + 1) % 10 == 0 or i + 1 == len(ALL):
            el = (time.time() - t0) / 60
            print(f'{label}: {i+1}/{len(ALL)} SACs, {el:.0f} min, ~{el/(i+1)*(len(ALL)-i-1):.0f} min left', flush=True)

## 10. Results table
One row per catalyst: ΔE/ΔG of every intermediate, Xu-style descriptors, `UL_distal`, `UL_alternating`,
the potential-determining step, ΔG*H, Ef, U_diss and the stage-1 stability flag.
`path_ok_distal = False` means at least one intermediate left the metal or broke during relaxation, so that U_L should not be trusted.
Columns ending in `_cal` are linearly calibrated to Xu's PBE-D3 scale (fit on the validation set).

In [ ]:
ads_df, sac_df = nm.analyse(f'{out(TASK)}/results.jsonl', refs='oc20' if TASK == 'oc20' else 'computed', use_d3=USE_D3)
sac_df = vl.add_udiss(sac_df, ref)
sac_df = vl.calibrate(sac_df, tables[BEST])
sac_df['model'] = f'{MODEL}:{BEST}'
f1, f2 = f'{WORK}/UMA_SAC_results_{BEST}.csv', f'{WORK}/UMA_structure_energies_{BEST}.csv'
sac_df.to_csv(f1, index=False); ads_df.to_csv(f2, index=False)
print('saved', f1, f2)
cols = [c for c in ['SAC', 'UL_distal', 'PDS_distal', 'path_ok_distal', 'UL_alternating', 'dG_N2_to_NNH_end', 'dG_H*', 'Ef_eV', 'U_diss_V', 'stable_stage1', 'status_bare'] if c in sac_df]
key = 'UL_distal' if 'UL_distal' in sac_df else 'dG_N2_to_NNH_end'
print(sac_df.sort_values(key, ascending=(key != 'UL_distal'))[cols].head(15).round(3).to_string(index=False))

In [ ]:
# Plot: UMA limiting potential vs HER descriptor, stable catalysts highlighted
if 'UL_distal' not in sac_df:
    sac_df['UL_distal'] = np.nan          # tier 3 not finished yet
ok = sac_df.get('path_ok_distal', pd.Series(True, index=sac_df.index)).fillna(False).astype(bool)
d = sac_df[ok].dropna(subset=['UL_distal', 'dG_H*'])          # only pathways where every intermediate stayed bound
fig, ax = plt.subplots(figsize=(6, 4.5))
for s, g in d.groupby('stable_stage1'):
    ax.scatter(g['dG_H*'], g['UL_distal'], s=22, label='stable (Ef<0, U_diss>0)' if s else 'unstable', alpha=.8)
for _, r in d.nlargest(8, 'UL_distal').iterrows():
    ax.annotate(r.SAC, (r['dG_H*'], r.UL_distal), fontsize=7)
ax.set_xlabel('ΔG*H (eV)'); ax.set_ylabel('U_L distal (V)'); ax.legend(); ax.set_title(f'{BEST}: NRR activity vs HER')
fig.tight_layout(); fig.savefig(f'{WORK}/UL_vs_dGH_{BEST}.png', dpi=150); plt.show()

**Send back** the files in `My Drive/NRR_UMA/`: `UMA_SAC_results_*.csv`, `UMA_structure_energies_*.csv`,
the `parity_*.png` plots and `results_*/results.jsonl`. They plug straight into the ML notebooks
(`NRR_activity_ML.ipynb`, `stability_ML_DasStyle.ipynb`) as a new, larger training set.